In [1]:
import os
import pickle
import numpy as np
import matplotlib.pyplot as plt
from utils.parse_results import get_results

# Subplots Shape
- 4 rows, 2 columns
    - F1, Acc
    - Active Region, Passive Region
    - Active Error Rate, Passive Error Rate
    - Active Error, Passive Error
- Each plot has
    - 4 groups (one for each dataset) 
        - each group has 6 bars (one for each method)


In [2]:
base_path = "../saved_data_cached/results/"

datasets = ["dsads", "rwhar", "pamap2", "opportunity"]
dataset_names = ["DSADS", "RWHAR", "PAMAP2", "OPPORTUNITY"]

architectures = ["tinyhar"]

metrics = [
    [r'$\mathbf{F1\ (\%) \uparrow}$', r'$\mathbf{Accuracy\ (\%) \uparrow}$'],
    [r'$\mathbf{Normalized\ Active\ Region\ \uparrow}$', r'$\mathbf{Normalized\ Passive\ Region\ \downarrow}$'],
    # Use \boldsymbol for the greek letters like epsilon
    [r'$\boldsymbol{\epsilon_{a}}\ \mathbf{(\%)\ \downarrow}$', r'$\boldsymbol{\epsilon_{p}}\ \mathbf{(\%)\ \downarrow}$'],
    [r'$\mathbf{Active\ Error\ (\%) \downarrow}$', r'$\mathbf{Passive\ Error\ (\%) \downarrow}$']
]

method_names = [
    r'$\mathbf{Cons\text{-}MSC\text{-}T}$',
    r'$\mathbf{Cons\text{-}MSC}$',
    r'$\mathbf{Cons\text{-}SSC}$',
    r'$\mathbf{Opp\text{-}MSC\text{-}T}$',
    r'$\mathbf{Opp\text{-}MSC}$',
    r'$\mathbf{Opp\text{-}SSC}$'
]

method_keys = ["conservative-asynchronous_multisensor_time_context",
               "conservative-asynchronous_multisensor",
               "conservative-asynchronous_single_sensor",
               "opportunistic-asynchronous_multisensor_time_context",
               "opportunistic-asynchronous_multisensor",
               "opportunistic-asynchronous_single_sensor",
                ]

In [3]:
plt.rcParams.update({
    # 1. Use STIX fonts - these are built-in to Matplotlib
    # and match the 'Times New Roman' / LaTeX serif aesthetic
    "font.family": "STIXGeneral",
    "mathtext.fontset": "stix",
    
    # 2. Force the font to be consistent across text and math
    "mathtext.rm": "STIXGeneral",
    "mathtext.it": "STIXGeneral:italic",
    "mathtext.bf": "STIXGeneral:bold",
    
    # 3. Size adjustments for professional legibility
    "axes.labelsize": 11,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.fontsize": 9,
    "axes.titlesize": 12,

    "axes.titleweight": "bold",    # Bold Subplot Titles
    "axes.labelweight": "bold",    # Bold X and Y Labels (e.g., "Performance")
    "font.weight": "bold",

    "ytick.labelsize": 12
})

# colors = [
#     '#deebf7', '#9ecae1', '#3182bd',
#     '#1f77b4', '#2ca02c', '#ff7f0e'
# ]

colors = [
    '#08306b', '#4292c6', '#c6dbef',  # Conservative (Blue)
    '#7f2704', '#f16913', '#fdd0a2'   # Opportunistic (Orange)
]

# policy_hatches = ['', '', '', '//', '//', '//']
# policy_hatches = ['xx', '//', '', 'xx', '//', '']
policy_hatches = ['', '', '', '', '', '']

In [4]:
import matplotlib.pyplot as plt
from matplotlib.ticker import MultipleLocator, AutoMinorLocator
import matplotlib.colors as mcolors  # Imported to parse any color format safely

# --- Helper Function for Text Color Contrast ---
def get_text_color_for_bar(color_input):
    """Determines if black or white text has better contrast against the given color."""
    try:
        # Convert color (HEX, name, RGB) to RGB tuple (0.0 to 1.0)
        rgb = mcolors.to_rgb(color_input)
        # Calculate relative luminance using standard W3C formula
        luminance = 0.2126 * rgb[0] + 0.7152 * rgb[1] + 0.0722 * rgb[2]
        # If the background is light, use black text; if dark, use white text
        return 'black' if luminance > 0.5 else 'white'
    except Exception:
        return 'black' # Fallback default

# Ensure rcParams are updated AT THE TOP for all figures
plt.rcParams.update({
    "font.family": "STIXGeneral",
    "mathtext.fontset": "stix",
    "font.weight": "bold",
    "axes.titleweight": "bold",
    "xtick.labelsize": 14,
    "ytick.labelsize": 12
})

# Loop through each row to create 4 separate figures
for r_i, metric_row in enumerate(metrics):
    # Create a new figure for each row (Ablation 1, 2, 3, 4)
    # 1 row, 2 columns
    fig, ax = plt.subplots(1, 2, figsize=(15, 2)) 
    
    for c_i, metric_col in enumerate(metric_row):
        result_dict = {dataset: [] for dataset in datasets}
        
        for dataset in datasets:
            for method in method_keys:
                # get all seed and subject runs
                result_path = os.path.join(base_path,dataset,architectures[0],method)
                result_logs = [os.path.join(result_path, filename) for filename in os.listdir(result_path)]
                
                # Has all metrics across each seed and subject
                results_list = []

                # for each seed
                for result_path in result_logs:
                    with open(result_path, 'rb') as f:
                        results = pickle.load(f)
                    
                    # get result for each user
                    subjects = results.keys()
                    vals = results.values()
                    results = [np.array(val) for val in vals]
                    results = np.stack(results)
                    results = np.concatenate([results,np.expand_dims(results[:,2]*results[:,4],1),np.expand_dims(results[:,3]*results[:,5],1)],axis=1)
                    
                    results_list.append(results)
                
                results_table = np.stack(results_list)
                subject_means = results_table.mean(axis=1)
                seed_std = subject_means.std(axis=0)
                
                result_dict[dataset].append(subject_means.mean(axis=0))

            result_dict[dataset] = np.stack(result_dict[dataset])
            result_dict[dataset] = 100*result_dict[dataset]
            result_dict[dataset][:,np.array([2,3])] = result_dict[dataset][:,np.array([2,3])]/100
           
        x = np.arange(len(datasets))
        total_width = 0.65
        width = total_width / len(method_keys)
        metric_i = 2 * r_i + (c_i % 2)

        for m_i in range(len(method_keys)):
            method_values = [result_dict[ds][m_i][metric_i] for ds in datasets]
            offset = (m_i * width) - (total_width / 2) + (width / 2)
            
            # 1. Capture the container object returned by ax.bar
            bars = ax[c_i].bar(x + offset, method_values, width, 
                               label=method_names[m_i],
                               edgecolor='black', 
                               linewidth=0.8, 
                               color=colors[m_i],
                               hatch=policy_hatches[m_i])
            
            # 2. Determine text color based on bar background color luminance
            text_color = get_text_color_for_bar(colors[m_i])
            
            # 3. Add labels within the bars
            # Use fmt='%.1f' for 1 decimal place, or adjust as needed for your values
            # print(metric_col)
            if r_i == 1:
                ax[c_i].bar_label(bars, 
                                label_type='center', 
                                rotation=90, 
                                color=text_color, 
                                fontweight='bold', 
                                fontsize=10,
                                fmt='%.2f') 
            else:
                ax[c_i].bar_label(bars, 
                              label_type='center', 
                              rotation=90, 
                              color=text_color, 
                              fontweight='bold', 
                              fontsize=10,
                              fmt='%.1f') 
            
        # --- Styling each subplot ---
        ax[c_i].set_title(metric_col, size=18) 
        ax[c_i].set_xticks(x)
        ax[c_i].set_xticklabels(dataset_names, fontweight='bold')
        
        # Grid settings
        ax[c_i].grid(True, which='major', axis='y', linestyle='--', linewidth=1)
        ax[c_i].grid(which='minor', axis='y', linestyle=':', linewidth=0.8, color='#d3d3d3')
        ax[c_i].set_axisbelow(True)

        # Default minor locator for those not specified above
        if r_i == 3:
            ax[c_i].yaxis.set_minor_locator(AutoMinorLocator(n=4))
            # if c_i == 1:
            #    ax[c_i].set_ylim(0, 20) 
        elif r_i == 1 and c_i == 1:
            ax[c_i].yaxis.set_minor_locator(AutoMinorLocator(n=4))
        elif r_i == 2 and c_i == 0:
            ax[c_i].yaxis.set_minor_locator(AutoMinorLocator(n=4))
        else:
            ax[c_i].yaxis.set_minor_locator(AutoMinorLocator(n=5))

    # --- Legend and Saving for the specific figure ---
    handles, labels = ax[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc='lower center', ncol=7, 
               bbox_to_anchor=(0.5, -0.2), prop={'size': 16, 'weight': 'bold'})
    
    plt.tight_layout(rect=[0, 0.05, 1, 1]) 
    plt.savefig(f'ablation_{r_i+1}.pdf', bbox_inches='tight')
    plt.close(fig)

In [5]:
import os
import pickle
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.ticker import AutoMinorLocator

# --- Setup Data and File Paths ---
base_path = "../saved_data_cached/results/"
datasets = ["dsads", "pamap2"]
dataset_names = ["DSADS", "PAMAP2"]
architectures = ["tinyhar","asyncformer"]

# Single row of 4 subplots
metrics = [
    r'$\mathbf{Normalized\ Passive\ Region\ \downarrow}$', 
    r'$\boldsymbol{\epsilon_{a}}\ \mathbf{(\%)\ \downarrow}$', 
    r'$\mathbf{F1\ (\%) \uparrow}$', 
    r'$\mathbf{Accuracy\ (\%) \uparrow}$'
]

# Mapping the 4 selected metrics back to their original array indices in your data:
# Active Region (row 1, col 0 -> idx 2), Active Epsilon (row 2, col 0 -> idx 4)
# F1 (row 0, col 0 -> idx 0), Accuracy (row 0, col 1 -> idx 1)
metric_indices = [3, 4, 0, 1]

# Only the 3 requested methods
method_names = [
    r'$\mathbf{Cons\text{-}MSC\text{-}T}$',
    r'$\mathbf{Heur\text{-}Transformer}$'
]

# Update these keys to match your actual folder names for the heuristic methods
method_keys = [
    "conservative-asynchronous_multisensor_time_context",
    "heuristic-asynchronous_transformer"
]

# --- Formatting & Colors ---
plt.rcParams.update({
    "font.family": "STIXGeneral",
    "mathtext.fontset": "stix",
    "font.weight": "bold",
    "axes.titleweight": "bold",
    "xtick.labelsize": 14,
    "ytick.labelsize": 12
})

# Custom 3-color palette (e.g., Deep Blue, Dark Orange, Muted Green)
colors = ['#08306b', '#E1D122']

def get_text_color_for_bar(color_input):
    """Determines if black or white text has better contrast against the given color."""
    try:
        rgb = mcolors.to_rgb(color_input)
        luminance = 0.2126 * rgb[0] + 0.7152 * rgb[1] + 0.0722 * rgb[2]
        return 'black' if luminance > 0.5 else 'white'
    except Exception:
        return 'black'

# --- Plotting Code ---
# Create 1 row with 4 subplots
fig, ax = plt.subplots(1, 4, figsize=(24, 4)) 

for c_i, metric_name in enumerate(metrics):
    metric_i = metric_indices[c_i]
    result_dict = {dataset: [] for dataset in datasets}
    
    for dataset in datasets:
        for method in method_keys:
            result_path = os.path.join(base_path, dataset, architectures[0], method)
            if method == "heuristic-asynchronous_transformer":
                result_path = os.path.join(base_path, dataset, architectures[1], method)
            
            # Catching cases where a path might not exist during development
            if not os.path.exists(result_path):
                print(f"Warning: Path missing {result_path}")
                result_dict[dataset].append(np.zeros(8)) # Fallback dummy array
                continue
                
            result_logs = [os.path.join(result_path, filename) for filename in os.listdir(result_path)]
            results_list = []

            for path in result_logs:
                with open(path, 'rb') as f:
                    results = pickle.load(f)
                
                vals = results.values()
                results = [np.array(val) for val in vals]
                results = np.stack(results)
                results = np.concatenate([
                    results,
                    np.expand_dims(results[:,2]*results[:,4], 1),
                    np.expand_dims(results[:,3]*results[:,5], 1)
                ], axis=1)
                
                results_list.append(results)
            
            results_table = np.stack(results_list)
            subject_means = results_table.mean(axis=1)
            result_dict[dataset].append(subject_means.mean(axis=0))

        result_dict[dataset] = np.stack(result_dict[dataset])
        result_dict[dataset] = 100 * result_dict[dataset]
        result_dict[dataset][:, np.array([2,3])] = result_dict[dataset][:, np.array([2,3])] / 100

    # Geometry for 3 bars per group
    x = np.arange(len(datasets))
    total_width = 0.65
    width = total_width / len(method_keys)

    for m_i in range(len(method_keys)):
        method_values = [result_dict[ds][m_i][metric_i] for ds in datasets]
        offset = (m_i * width) - (total_width / 2) + (width / 2)
        
        bars = ax[c_i].bar(x + offset, method_values, width, 
                           label=method_names[m_i],
                           edgecolor='black', 
                           linewidth=0.8, 
                           color=colors[m_i])
        
        text_color = get_text_color_for_bar(colors[m_i])
        
        # Format decimal precision depending on the metric type
        fmt_str = '%.2f' if metric_i in [2, 3] else '%.1f'
        
        ax[c_i].bar_label(bars, 
                          label_type='center', 
                          rotation=90, 
                          color=text_color, 
                          fontweight='bold', 
                          fontsize=10,
                          fmt=fmt_str) 
        
    # --- Styling each subplot ---
    ax[c_i].set_title(metric_name, size=16) 
    ax[c_i].set_xticks(x)
    ax[c_i].set_xticklabels(dataset_names, fontweight='bold', rotation=15) # Small rotation if labels crowd
    
    ax[c_i].grid(True, which='major', axis='y', linestyle='--', linewidth=1)
    ax[c_i].grid(which='minor', axis='y', linestyle=':', linewidth=0.8, color='#d3d3d3')
    ax[c_i].set_axisbelow(True)
    ax[c_i].yaxis.set_minor_locator(AutoMinorLocator(n=4))

# --- Global Legend and Layout ---
handles, labels = ax[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='lower center', ncol=3, 
           bbox_to_anchor=(0.5, -0.15), prop={'size': 14, 'weight': 'bold'})

plt.tight_layout(rect=[0, 0.05, 1, 1]) 
plt.savefig('repurposed_metrics_plot.pdf', bbox_inches='tight')
plt.close(fig)